In [0]:
patients_bronze = spark.table("workspace.healthcare.bronze_patients")
display(patients_bronze)

patient_id,first_name,last_name,gender,date_of_birth,city,registration_date
P001,Rahul,Sharma,Male,1985-04-12,Hyderabad,2025-01-10
P002,Priya,Rao,Female,1990-07-21,Mumbai,2025-01-15
P003,Arjun,Reddy,Male,1978-02-10,Hyderabad,2025-01-20
P004,Sneha,Patel,Female,1995-11-05,Bangalore,2025-02-01
P005,Vikram,Singh,Male,1982-08-18,Delhi,2025-02-05
P006,Ananya,Iyer,Female,2000-03-15,Chennai,2025-02-10
P007,Kiran,Kumar,Male,1975-06-25,Hyderabad,2025-02-15
P008,Meena,Das,Female,1988-09-12,Kolkata,2025-02-20
P009,Rohit,Verma,Male,1992-12-01,Delhi,2025-03-01
P010,Lakshmi,Nair,Female,1980-01-30,Kochi,2025-03-05


In [0]:
patients_bronze.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: string (nullable = true)



In [0]:
from pyspark.sql.functions import(
    col,
    sum,
    when
)

In [0]:
null_check = patients_bronze.select(
    [
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in patients_bronze.columns
    ]
)

display(null_check)

patient_id,first_name,last_name,gender,date_of_birth,city,registration_date
0,0,0,0,0,0,0


In [0]:
patients_clean = patients_bronze.dropDuplicates(["patient_id"])

In [0]:
print("Records after removing duplicates:",patients_clean.count())

Records after removing duplicates: 10


In [0]:
patients_clean = patients_clean.filter(col("patient_id").isNotNull())

In [0]:
from pyspark.sql.functions import (
    trim,
    initcap
)

In [0]:
# Skipping full_name transformation - column does not exist in source data

In [0]:
patients_clean = patients_clean.withColumn(
    "first_name",
    initcap(trim(col("first_name")))
)

In [0]:
patients_clean = patients_clean.withColumn(
    "last_name",
    initcap(trim(col("last_name")))
)

In [0]:
patients_clean = patients_clean.withColumn(
    "city",
    initcap(trim(col("city")))
)

In [0]:
from pyspark.sql.functions import upper

In [0]:
patients_clean = patients_clean.withColumn(
    "gender",
    upper(trim(col("gender")))
)

In [0]:
patients_clean = patients_clean.withColumn(
    "gender",
    upper(trim(col("gender")))
)

In [0]:
from pyspark.sql.functions import to_date

patients_clean = patients_clean.withColumn(
    "registration_date",
    to_date(col("registration_date"))
)

In [0]:
patients_clean = patients_clean.withColumn(
    "registration_date",
    to_date(col("registration_date"))
)

In [0]:
from pyspark.sql.functions import year, current_date

# Calculate age from date_of_birth
patients_with_age = patients_clean.withColumn(
    "age",
    year(current_date()) - year(to_date(col("date_of_birth")))
)

invalid_age = patients_with_age.filter(
    (col("age") < 0) |
    (col("age") > 120)
)

display(invalid_age)

patient_id,first_name,last_name,gender,date_of_birth,city,registration_date,age


In [0]:
from pyspark.sql.functions import current_timestamp

patients_final = patients_with_age.withColumn(
    "processed_timestamp",
    current_timestamp()
)

display(patients_final)

patient_id,first_name,last_name,gender,date_of_birth,city,registration_date,age,processed_timestamp
P001,Rahul,Sharma,MALE,1985-04-12,Hyderabad,2025-01-10,41,2026-09-01T06:10:01.359Z
P002,Priya,Rao,FEMALE,1990-07-21,Mumbai,2025-01-15,36,2026-09-01T06:10:01.359Z
P003,Arjun,Reddy,MALE,1978-02-10,Hyderabad,2025-01-20,48,2026-09-01T06:10:01.359Z
P004,Sneha,Patel,FEMALE,1995-11-05,Bangalore,2025-02-01,31,2026-09-01T06:10:01.359Z
P005,Vikram,Singh,MALE,1982-08-18,Delhi,2025-02-05,44,2026-09-01T06:10:01.359Z
P006,Ananya,Iyer,FEMALE,2000-03-15,Chennai,2025-02-10,26,2026-09-01T06:10:01.359Z
P007,Kiran,Kumar,MALE,1975-06-25,Hyderabad,2025-02-15,51,2026-09-01T06:10:01.359Z
P008,Meena,Das,FEMALE,1988-09-12,Kolkata,2025-02-20,38,2026-09-01T06:10:01.359Z
P009,Rohit,Verma,MALE,1992-12-01,Delhi,2025-03-01,34,2026-09-01T06:10:01.359Z
P010,Lakshmi,Nair,FEMALE,1980-01-30,Kochi,2025-03-05,46,2026-09-01T06:10:01.359Z


In [0]:
patients_final.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- age: integer (nullable = true)
 |-- processed_timestamp: timestamp (nullable = false)



In [0]:
patients_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.healthcare.silver_patients"
    )

In [0]:
patients_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.healthcare.silver_patients"
    )

In [0]:
silver_patients = spark.table(
    "workspace.healthcare.silver_patients"
)

display(silver_patients)

patient_id,first_name,last_name,gender,date_of_birth,city,registration_date,age,processed_timestamp
P004,Sneha,Patel,FEMALE,1995-11-05,Bangalore,2025-02-01,31,2026-09-01T06:11:57.230Z
P009,Rohit,Verma,MALE,1992-12-01,Delhi,2025-03-01,34,2026-09-01T06:11:57.230Z
P006,Ananya,Iyer,FEMALE,2000-03-15,Chennai,2025-02-10,26,2026-09-01T06:11:57.230Z
P010,Lakshmi,Nair,FEMALE,1980-01-30,Kochi,2025-03-05,46,2026-09-01T06:11:57.230Z
P003,Arjun,Reddy,MALE,1978-02-10,Hyderabad,2025-01-20,48,2026-09-01T06:11:57.230Z
P005,Vikram,Singh,MALE,1982-08-18,Delhi,2025-02-05,44,2026-09-01T06:11:57.230Z
P001,Rahul,Sharma,MALE,1985-04-12,Hyderabad,2025-01-10,41,2026-09-01T06:11:57.230Z
P008,Meena,Das,FEMALE,1988-09-12,Kolkata,2025-02-20,38,2026-09-01T06:11:57.230Z
P002,Priya,Rao,FEMALE,1990-07-21,Mumbai,2025-01-15,36,2026-09-01T06:11:57.230Z
P007,Kiran,Kumar,MALE,1975-06-25,Hyderabad,2025-02-15,51,2026-09-01T06:11:57.230Z
